# 03 · Data: collection, parsing, dedupe, sanity counts

Collection (needs `CR_API_KEY` in `.env`, key registered for your current IP):
```
python -m crdeck gate      # /cards, leaderboard, one battle log -> fill docs/api-notes.md
python -m crdeck collect   # top players' logs for config window (cached; resumable)
python -m crdeck build     # raw + labels -> data/interim
```
This notebook only reads what `build` produced. If the API failed or nothing was collected, it says so and stops.

In [ ]:
import os
import json
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

DATA_SOURCE = os.environ.get("CRDECK_DATA_SOURCE", "real")  # "real" (default) or "synthetic"
from crdeck.config import Paths
paths = Paths.for_source(DATA_SOURCE)
print(paths.banner())

In [ ]:
from crdeck.pipeline import load_interim
battles, dib, decks, counts = load_interim(paths)
print(json.dumps({k: v for k, v in counts.items() if k != "card_attach_report"}, indent=1, default=str))

## Status counts
`OK` battles feed the models. Mirrors and draws are stored but excluded; `UNKNOWN_CARD` battles wait for labels.

In [ ]:
pd.Series(counts["status"]).rename("battles").to_frame()

In [ ]:
if counts["unknown_card_ids"]:
    print("Card ids missing from the label table (add them to data/labels/cards.csv):", counts["unknown_card_ids"])
else:
    print("No unknown cards.")

## Battles per day (UTC) and battle types

In [ ]:
ok = battles[battles.status == "OK"].copy()
ok["day"] = pd.to_datetime(ok["battle_time"]).dt.strftime("%Y-%m-%d")
print(ok.groupby("day").size().to_string())
print(battles["battle_type"].value_counts().to_string())

## Sanity checks

In [ ]:
assert battles["battle_key"].is_unique, "duplicate battles after dedupe"
assert (dib.groupby("battle_key").size() == 2).all()
print("battles after dedupe:", len(battles), "| duplicates removed:", counts["duplicates_removed"])
print("distinct decks:", len(decks), "| decks seen once:", int((decks.appearances == 1).sum()))
print("decks with hero-capable cards in unknown form:", int(decks.form_unknown.sum()))
lg = dib["level_deficit"].dropna()
print("per-side mean level deficit (maxLevel - level): ", lg.describe().round(2).to_dict())